# 6. Gathering và dựng hierarchy theo version

## 6.1 Chọn compilation root

Parser đọc các file TeX và tìm root có `documentclass/documentstyle`. Nếu chỉ có một ứng viên,
root đó được dùng. Khi nhiều ứng viên, cấu hình override chỉ được thêm sau đối chiếu nội dung
title và vai trò main/supplementary/template. Trường hợp chưa đủ bằng chứng vẫn báo lỗi.

Gathering duyệt `input/include` theo thứ tự, bỏ comment và phát hiện cycle hoặc đường dẫn
thoát thư mục version. Input thông thường được tìm từ thư mục biên dịch; nếu không tồn tại,
triển khai còn thử đường dẫn tương đối của file con để tương thích một số bố cục nguồn.
File không được include không thuộc compilation path. Parser không chạy TeX hoặc giải mã
mọi macro như một trình biên dịch đầy đủ.

## 6.2 Quy tắc hierarchy

Section stack xác định parent theo mức chapter/section/subsection. List và item tạo các
container riêng. Paragraph được chia tại dòng trống ngoài nhóm/lệnh TeX và toán; câu được
chia ở dấu kết thúc phù hợp, có bảo vệ command, URL, số thập phân và một số viết tắt.
Block math và figure/table là leaf riêng. Acknowledgements và appendix được giữ; bibliography
được tách khỏi hierarchy để xử lý references.

Source map liên kết khoảng ký tự trong bản gathered với file và dòng gốc. Thông tin này
giúp đối chiếu khi một version partial; cây có cấu trúc hợp lệ vẫn có thể thiếu nội dung
do cú pháp chưa được hỗ trợ.

## 6.3 Ví dụ thực thi từ source ghim

Ví dụ sau là TeX **tổng hợp để minh họa**, không phải nội dung hay số liệu từ corpus.
Nó chạy chính các hàm gathering/hierarchy của code repository, không gọi API.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print(f"Snapshot: {scope['papers']} paper / {scope['versions']} version; báo cáo ngày {data['report_date']}")

In [ ]:
import tempfile
from parser.gather import gather
from parser.hierarchy import parse_hierarchy
main = r"""\documentclass{article}
\title{Illustrative document}
\begin{document}
\input{sections/body}
\section*{Acknowledgements}
We thank the example team.
\appendix
\section{Additional detail}
The appendix is retained.
\begin{thebibliography}{9}
\bibitem{demo} A. Example. A sample title. 2020.
\end{thebibliography}
\end{document}
"""
body = r"""\section{Introduction}
First sentence. See Fig. 1. The value is $1.5$.
\begin{itemize}\item First point.\item Second point.\end{itemize}
\input{tables/result}
\begin{equation}x+y=1\end{equation}
\begin{figure}\caption{Illustrative figure.}\end{figure}
"""
table = r"\begin{table}\caption{Illustrative table.}\end{table}"
with tempfile.TemporaryDirectory() as temp:
    root = Path(temp); (root/'sections').mkdir(); (root/'tables').mkdir()
    (root/'main.tex').write_text(main,encoding='utf-8')
    (root/'sections/body.tex').write_text(body,encoding='utf-8')
    (root/'tables/result.tex').write_text(table,encoding='utf-8')
    (root/'unused.tex').write_text('This file is not included.',encoding='utf-8')
    gathered = gather(root); tree = parse_hierarchy(gathered.text)
    print('Compilation path:',gathered.files)
    print('Diagnostics:',gathered.diagnostics + tree.diagnostics)
    display(pd.DataFrame([{'node':n.node_id,'parent':n.parent_id,'kind':n.kind,'content':n.content[:100]} for n in tree.nodes]))

## 6.4 Kết quả trên snapshot và các sửa đổi đã đối chiếu

Bảng trước/sau bên dưới, nếu có, dùng hai lượt parser trên cùng input snapshot. Các thay đổi
gồm sửa cách tìm include từ compilation directory và cấu hình root cho những ca đã đối chiếu.
So sánh này chứng minh tác động đến trạng thái xử lý trên tập đã biết; không phải thí nghiệm
lặp độc lập để kết luận tốc độ hoặc khả năng tổng quát cho toàn arXiv.

In [ ]:
status = pd.Series(m2['status_counts'],name='versions'); display(status.to_frame())
if m2.get('before_fixes_status'):
    comparison = pd.DataFrame({'Trước sửa':m2['before_fixes_status'],'Sau sửa':m2['status_counts']}).fillna(0).astype(int)
    display(comparison)
    comparison.plot.bar(title='Trạng thái trước/sau trên cùng snapshot',ylabel='Số version'); plt.tight_layout(); plt.show()
tex = m2.get('versions_with_tex',scope['versions']-m2['status_counts'].get('empty_no_tex',0)-m2['status_counts'].get('error_missing_source',0))
success = m2['status_counts'].get('success',0); partial = m2['status_counts'].get('partial',0)
display(pd.DataFrame([
    {'metric':'Recorded / expected','value':sum(r['recorded_versions'] for r in m2['coverage'])/scope['versions']},
    {'metric':'Success / all versions','value':success/scope['versions']},
    {'metric':'Success / versions with TeX','value':success/tex},
    {'metric':'Success + partial / versions with TeX','value':(success+partial)/tex},
]))

## 6.5 Coverage từng publication

Nhãn P001…P073 là nhãn công khai cho bảng coverage, không phải arXiv ID. `recorded_versions`
bao gồm success, partial, empty và error. `parsed_version_coverage` tính success + partial;
`successful_version_coverage` chỉ tính success. Version empty có bằng chứng là một kết quả
hợp lệ của pipeline nhưng không có parsed elements.

In [ ]:
columns=['paper','expected_versions','recorded_versions','versions_with_tex','success_versions',
         'partial_versions','empty_versions','error_versions','parsed_version_coverage','successful_version_coverage']
with pd.option_context('display.max_rows',None,'display.max_columns',None):
    display(pd.DataFrame(m2['coverage'])[columns])
diag = pd.Series(m2['diagnostic_categories'],name='versions_with_diagnostic')
display(diag.to_frame())
diag.plot.barh(title='Nhóm diagnostic — một version có thể có nhiều nhóm',xlabel='Số version')
plt.tight_layout(); plt.show()

## 6.6 Các giới hạn còn lại

Missing bibliography, include động và môi trường tự định nghĩa vẫn có thể làm version
partial. Một version còn nhiều root cùng title được giữ lỗi thay vì đoán theo tên hoặc
kích thước file. Một version còn thiếu nguồn được ghi `error_missing_source`.
Việc có manifest cho mọi version chỉ xác nhận khả năng theo dõi trạng thái, không chứng
minh parser đã tái hiện đầy đủ mọi nội dung biên dịch.

## Tài liệu và mã tham chiếu

- [Đặc tả M2](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API: cấu trúc truy vấn và Atom](https://info.arxiv.org/help/api/user-manual.html).
- [Quy định truy cập arXiv](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã triển khai: [src/parser/gather.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/parser/gather.py).
- Mã triển khai: [src/parser/latex.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/parser/latex.py).
- Mã triển khai: [src/parser/hierarchy.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/parser/hierarchy.py).
- Mã triển khai: [src/parser/main.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/src/parser/main.py).
- Mã triển khai: [configs/submission_parser_roots_20261106.json](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/6cd8dac6d1491083286d538dd7fffacf66f4cc4c/configs/submission_parser_roots_20261106.json).